# Invoice parsing tests by file type

Tests for each format in `data/invoices/`: **JSON**, **TXT**, **CSV** (row-based and key-value), **XML**, **PDF**.

In [15]:
from pathlib import Path

from invoice_parsers import (
    parse_invoice,
    parse_invoices,
    parse_json,
    parse_txt,
    parse_csv_row_based,
    parse_csv_key_value,
    parse_xml,
    parse_pdf,
)

INVOICES_DIR = Path("./data/invoices")

In [16]:
# --- TEST: JSON (helper + main) ---
path = INVOICES_DIR / "invoice_1004.json"
data = parse_invoice(path)  # main dispatcher
print("parser_used:", data["parser_used"])
print("Vendor:", data["vendor"], "| Total:", data["total"])
print("Line items:", data["line_items"])

Vendor: {'name': 'Precision Parts Ltd.', 'address': '742 Evergreen Terrace, Springfield, IL 62704'}
Total: 1890.0
Line items: [{'item': 'WidgetA', 'quantity': 3, 'unit_price': 250.0}, {'item': 'WidgetB', 'quantity': 2, 'unit_price': 500.0}]


In [17]:
# --- TEST: TXT (helper + main) ---
path = INVOICES_DIR / "invoice_1008.txt"
data = parse_invoice(path)
print("parser_used:", data["parser_used"])
print("raw_text (first 400 chars):", (data.get("raw_text") or "")[:400])

Lines: 17
From: billing@noproduct.biz
To: accounts@acmecorp.com
Subject: Invoice Attached - Please Process

Hi,

Please find the invoice details below:

  Vendor: NoProd Industries
  Invoice: INV-1008
  Date: 2026-01-10
  Due: 2026-01-20

  Items Ordered:
    - SuperGizmo       x12     $400.00 each
    - MegaSprocket     x6      $850.00 each

  Total: $9,900.00

Please process at your earliest convenience.

Best regards,
Accounts Receivable
NoProd Industries



In [18]:
# --- TEST: CSV row-based (main uses fallback: tries row-based then key-value) ---
path = INVOICES_DIR / "invoice_1015.csv"
data = parse_invoice(path)
print("parser_used:", data["parser_used"])
print("Vendor:", data["vendor"], "| Total:", data["total"])
print("Line items:", data["line_items"])

Line items (row-based CSV):
      Item   Qty Unit Price  Line Total
0  WidgetA  10.0     250.00      2500.0
1  WidgetB   5.0     500.00      2500.0
2  GadgetX   2.0     750.00      1500.0

Invoice header: {'Invoice Number': 'INV-1015', 'Vendor': 'Reliable Components Inc.', 'Date': '2026-01-29', 'Due Date': '2026-02-28'}


In [19]:
# --- TEST: CSV key-value (main tries row-based first, then key-value) ---
path = INVOICES_DIR / "invoice_1006.csv"
data = parse_invoice(path)
print("parser_used:", data["parser_used"])
print("Vendor:", data["vendor"], "| Total:", data["total"])
print("Line items:", data["line_items"])

Key-value CSV - header: {'invoice_number': 'INV-1006', 'vendor': 'Acme Industrial Supplies', 'date': '2026-01-25', 'due_date': '2026-02-10', 'subtotal': '2750.00', 'tax': '0.00', 'total': '2750.00', 'payment_terms': 'Net 15'}
Line items (reconstructed): [{'item': 'WidgetA', 'qty': 5, 'unit_price': 250.0}, {'item': 'WidgetB', 'qty': 3, 'unit_price': 500.0}]


In [20]:
# --- TEST: XML (helper + main) ---
path = INVOICES_DIR / "invoice_1014.xml"
data = parse_invoice(path)
print("parser_used:", data["parser_used"])
print("Vendor:", data["vendor"], "| Total:", data["total"])
print("Line items:", data["line_items"])

Header: {'invoice_number': 'INV-1014', 'vendor': 'TechParts International', 'date': '2026-01-26', 'due_date': '2026-02-26', 'currency': 'EUR'}
Line items: [{'name': 'WidgetA', 'quantity': 4, 'unit_price': 225.0}, {'name': 'WidgetB', 'quantity': 6, 'unit_price': 475.0}]
Total: 4125.00


In [21]:
# --- TEST: PDF (main uses fallback: pdfplumber then PyMuPDF) ---
path = INVOICES_DIR / "invoice_1011.pdf"
data = parse_invoice(path)
print("parser_used:", data["parser_used"])
print("raw_text (first 500 chars):", (data.get("raw_text") or "")[:500])

PDF text (pdfplumber), first 600 chars:
INVOICE
Invoice Number: INV-1011
Vendor: Summit Manufacturing Co.
Date: 2026-01-20
Due Date: 2026-02-20
Item Qty Unit Price Amount
WidgetA 6 $250.00 $1,500.00
WidgetB 3 $500.00 $1,500.00
Total: $3,000.00


In [ ]:
# --- MAIN: parse multiple invoice paths (auto parser + fallbacks) ---
paths = [
    INVOICES_DIR / "invoice_1004.json",
    INVOICES_DIR / "invoice_1008.txt",
    INVOICES_DIR / "invoice_1015.csv",
    INVOICES_DIR / "invoice_1006.csv",
    INVOICES_DIR / "invoice_1014.xml",
    INVOICES_DIR / "invoice_1011.pdf",
]
results = parse_invoices(paths)
for path, data in zip(paths, results):
    print(f"{path.name}: parser={data['parser_used']} | vendor={data.get('vendor')} | total={data.get('total')}")